# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
ds_metadata = dataset.metadata

print(f"{ds_metadata.name}: {ds_metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Display available record sets with their names and IDs
print('Record Sets:')
for rs in ds_metadata.record_sets:
    print(f"  - Name: {rs.name}, @id: {rs.id}")

# For each record set, list the fields and columns
for rs in ds_metadata.record_sets:
    print(f"\nRecord set: {rs.name} | @id: {rs.id}")
    print("Fields:")
    for field in rs.fields:
        print(f"  - Name: {field.name}, @id: {field.id}, dataType: {field.data_type}")
        if hasattr(field, 'column') and field.column is not None:
            print(f"    Column: {field.column.name if hasattr(field.column, 'name') else field.column}, @id: {field.column.id if hasattr(field.column, 'id') else field.column}")
    print("----")

## 2.1 Sample Records
Preview a few rows from a record set by `@id`. Replace `<record_set_id>` with an actual `@id` from the list above.


In [ ]:
# Pick the primary record set id from those printed above
# Example (replace this with the id you want if different):
primary_record_set_id = ds_metadata.record_sets[0].id if ds_metadata.record_sets else None
if primary_record_set_id:
    for idx, record in enumerate(dataset.records(record_set=primary_record_set_id)):
        print(record)
        if idx > 4:
            break
else:
    print("No record sets available in this dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set IDs
record_set_ids = [rs.id for rs in ds_metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id={record_set_id} with shape: {df.shape}")

# Show the columns of the main record set
if record_set_ids:
    print(f"\nColumns available in main record set ({record_set_ids[0]}):")
    print(dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()
else:
    print("No record sets to load data from.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Select the record set and numeric field for EDA
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id] if record_set_id else None

numeric_field_id = None
# Try to select a numeric field automatically from metadata
if record_set_id and df is not None:
    rs = next((r for r in ds_metadata.record_sets if r.id == record_set_id), None)
    numeric_fields = [field for field in rs.fields if field.data_type in ['Number', 'Float', 'Integer'] and field.id in df.columns]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        numeric_field_id = numeric_field.id
        print(f"Using numeric field: {numeric_field.name} (@id: {numeric_field_id})")
    else:
        print("No numeric field found in this record set.")
else:
    print("No data-frame available for EDA.")

if numeric_field_id and record_set_id:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0

    # Filter records above the threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (mean value):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a non-numeric field
    group_fields = [field for field in rs.fields if field.data_type == 'Text' and field.id in df.columns]
    if group_fields:
        group_field = group_fields[0].id
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No group (text) field found in this record set for grouping.")
else:
    print("Unable to identify numeric fields in record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization: Histogram of the selected numeric field
if record_set_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If there is a group field, display boxplot
    if group_fields:
        group_field_id = group_fields[0].id
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset comprises protein analysis results from mass spectrometry of extracellular vesicles from human mast cells.
- Numeric fields such as coverage percentages, peptide counts, or normalized abundances are available for quantitative analysis.
- EDA shows basic statistics and the possibility to group or filter by protein identifiers or classes, supporting biomarker discovery or expression studies.
- For comprehensive molecular analysis, further domain-specific processing can be performed.